<a href="https://colab.research.google.com/github/nguyenanhtienabcd/M09_PROJ_Text-Guided_Image_Generation_using_Conditional_Flow_Matching/blob/feature%2Fproject-text-guided/m09_project_Text_Guided_Image_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install libs
!pip install -q datasets torchcfm

# Dataset
from datasets import load_dataset

ds = load_dataset("bahjat-kawar/tedbench", split="val")

# Text Encoder
import torch
from sentence_transformers import SentenceTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
text_encoder = SentenceTransformer("all-mpnet-base-v2").to(device)


In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# tạo một lớp lấy dữ liệu
class TextGuidedImageGenerationDataset(Dataset):
    def __init__(self, dataset, text_encoder, device, transform=None):
        self.dataset = dataset
        self.images = images
        self.text_encoder = text_encoder
        self.transform = transform
        self.device = device
        self.original_images = dataset['original_image']
        self.captions = dataset['caption']
        self.edited_image = dataset['edited_image']
        self.embed_captions = text_encoder.encode(
            self.captions, convert_to_tensor=True, device=device
        )

    def __getitem__(self, idx):
        # get an image (covert image to tensor)
        original_image = self.original_images[idx]
        original_image = self.transform(original_image)

        edited_image = self.edited_image[idx]
        edited_image = self.transform(edited_image)

        # get a text (convert text to tensor)
        caption = self.captions[idx]
        caption_embedding = self.embed_captions[idx]

        return {
            "original_image": original_image,
            "edited_image": edited_image,
            "caption": caption,
            "caption_embedding": caption_embedding,
        }

    def __len__(self):
        return len(self.dataset)

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()]
)

train_loader = Dataloader(train_ds, batch_size=256, shuffle=True)

In [ ]:
# xây dựng mô hình time embedding
import math
import torch.nn as nn
from torchcfm.models.unet import UNetModel

def timestep_embedding(timesteps, dim, max_period=10000):
    """Create sinusoidal timestep embeddings.

    :param timesteps: a 1-D Tensor of N indices, one per batch element. These may be fractional.
    :param dim: the dimension of the output.
    :param max_period: controls the minimum frequency of the embeddings.
    :return: an [N x dim] Tensor of positional embeddings.
    """
    half = dim // 2
    freqs = torch.exp(
        -math.log(max_period)
        * torch.arange(start=0, end=half, dtype=torch.float32, device=timesteps.device)
        / half
    )
    #timesteps[:, None]: thêm 1 chiều (2, 1)
    #freqs[None, :]: thêm batch dimension (1, 3) → nhân broadcast để có (2, 3)
    args = timesteps[:, None].float() * freqs[None]

    embedding = torch.cat([torch.cos(args), torch.sin(args)], dim=-1)
    if dim % 2:
        embedding = torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
    return embedding


In [ ]:
import torch
import torch.nn as nn

# Giả sử UNetModel, timestep_embedding đã được định nghĩa từ trước

class UNetModelWithTextEmbedding(UNetModel):
    def __init__(self, dim, num_channels, num_res_blocks, embedding_dim, *args, **kwargs):
        super().__init__(dim, num_channels, num_res_blocks, *args, **kwargs)

        self.image_encoder = nn.Sequential(
            nn.Conv2d(3, num_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(num_channels, num_channels * 2, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(num_channels * 2, num_channels * 4, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.embedding_layer = nn.Linear(embedding_dim, num_channels * 4)
        self.fc = nn.Linear(num_channels * 12, num_channels * 4)

    def forward(self, t, x, text_embeddings=None, original_image=None):
        """Apply the model to an input batch, incorporating text embeddings."""
        # Bắt đầu xử lý embedding thời gian
        timesteps = t

        # Đảm bảo timesteps là tensor 1 chiều
        while timesteps.dim() > 1:
            timesteps = timesteps[:, 0]
        if timesteps.dim() == 0:
            timesteps = timesteps.repeat(x.shape[0])

        hs = []
        emb = self.time_embed(timestep_embedding(timesteps, self.model_channels))

        # ghép nối text + image + time ( sinh ảnh có điều kiện đầu vào)
        if (text_embeddings is not None) and (original_image is not None):
            text_embedded = self.embedding_layer(text_embeddings)
            image_embedded = self.image_encoder(original_image).squeeze(2).squeeze(2)
            emb = torch.cat([emb, text_embedded, image_embedded], dim=1)
            emb = self.fc(emb)


        # Ép kiểu dữ liệu đầu vào (x) nếu cần thiết (float16, float32, ...)
        h = x.type(self.dtype)

        # Đi qua các block đầu vào của UNet (downsampling)
        for module in self.input_blocks:
            h = module(h, emb)
            hs.append(h)  # Lưu lại để skip connection sau

        # Đi qua block giữa của UNet (bottleneck)
        h = self.middle_block(h, emb)

        # Đi qua các block đầu ra (upsampling), dùng skip connection
        for module in self.output_blocks:
            h = torch.cat([h, hs.pop()], dim=1)  # Ghép với feature tương ứng từ input_blocks
            h = module(h, emb)

        # Trả lại về kiểu ban đầu
        h = h.type(x.dtype)
        return self.out(h)  # Trả về ảnh đầu ra


In [ ]:
from tqdm import tqdm

# Khởi tạo mô hình UNet có tích hợp text embedding
model = UNetModelWithTextEmbedding(
    dim=(3, 256, 256),               # Input: ảnh RGB 256x256
    num_channels=32,
    num_res_blocks=1,
    embedding_dim=768             # Dimensionality của text embedding từ SentenceTransformer
).to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters())
n_epochs = 1000 #20000  # Số epoch huấn luyện

# Bắt đầu huấn luyện
for epoch in tqdm(range(n_epochs)):
    losses = []

    for batch in train_loader:
        optimizer.zero_grad()

        # Lấy ảnh và embedding caption từ batch
        original_image = batch["original_image"].to(device)
        edited_image = batch["edited_image"].to(device)
        text_embeddings = batch["caption_embedding"].to(device)

        x1 = edited_image

        # Tạo noise ngẫu nhiên cùng shape với x1
        x0 = torch.randn_like(x1).to(device)

        # Lấy ngẫu nhiên timestep t ∈ [0, 1] cho mỗi mẫu trong batch
        t = torch.rand(x0.shape[0], 1, 1, 1).to(device)

        # Tạo điểm trung gian xt trên đường thẳng từ x0 → x1
        xt = x1 * t + (1 - t) * x0

        # Đạo hàm vận tốc (vector) cần học
        ut = x1 - x0  # vector hướng từ x0 đến x1
        t = t.squeeze()  # loại bỏ các chiều dư thừa

        # Mô hình dự đoán vt từ xt, t và caption
        vt = model(t, xt, text_embeddings=text_embeddings)

        # Tính loss giữa vector thật và vector dự đoán
        loss = torch.mean(((vt - ut) ** 2))  # MSE loss

        # Lan truyền ngược
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    # Tính loss trung bình mỗi epoch
    avg_loss = sum(losses) / len(losses)

    # In log mỗi 500 epoch
    if (epoch + 1) % 500 == 0:
        print(f"Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}")


In [ ]:
model.eval()  # Đưa model sang chế độ đánh giá (không dropout, batchnorm cố định)

# Hàm giải phương trình vi phân đạo hàm bậc nhất bằng Euler method
def euler_method(model, text_embedding, t_steps, dt, noise):
    y = noise             # Khởi tạo từ ảnh nhiễu (giống ảnh noise đầu vào của diffusion)
    y_values = [y]        # Danh sách để lưu các bước trung gian (ảnh từ t=0 → t=1)

    with torch.no_grad():  # Không cần tính gradient trong lúc suy luận
        for t in t_steps[1:]:
            t = t.reshape(-1, )  # Đưa t về dạng [batch_size] để tương thích model
            dy = model(t.to(device), y, text_embeddings=text_embedding)  # Dự đoán vận tốc
            y = y + dy * dt       # Bước Euler: cập nhật ảnh
            y_values.append(y)    # Lưu lại ảnh sau bước này

    return torch.stack(y_values)  # Trả lại toàn bộ quá trình chuyển động ảnh

# ======================
# Inference bắt đầu ở đây
# ======================

# Khởi tạo ảnh noise ngẫu nhiên (input ban đầu)
sample = train_ds[5]
original_image = sample["original_image"].unsqueeze(0).to(device)
edited_image = sample["edited_image"].unsqueeze(0).to(device)
text_embedding = sample["caption_embedding"].unsqueeze(0).to(device)
noise = torch.randn(1, 3, 256, 256).to(device)

# Khởi tạo các bước thời gian (100 bước đều từ 0 → 1)
t_steps = torch.linspace(0, 1, 100, device=device)

# Khoảng thời gian giữa các bước (dt)
dt = t_steps[1] - t_steps[0]

# Giải phương trình vi phân để "di chuyển ảnh" từ noise → ảnh thật (theo chiều thời gian)
results = euler_method(model, text_embedding, t_steps, dt, noise)